# Runtime and Scalabilty Test

Benchmarking to report runtime and scalabiltiy metrics.


In [ ]:
%%capture
# install necessary dependencies (this can take a few minutes)
%pip install splitpea
%pip install gseapy  # needed for enrichment functionality

# additional mapping file for organism id conversions
!wget https://raw.githubusercontent.com/ylaboratory/splitpea/master/reference/hsa_mapping_all.txt

# Tabix is used under-the-hood for some of the functions and should be installed before running
# install directly via the command line or with conda
%conda install -y -c bioconda tabix

In [ ]:
import splitpea
import pandas as pd
import gseapy as gp
import os, random, time
from pathlib import Path


In this example, we use the rMATS-turbo example data for prostate cancer cell lines stored [here](https://zenodo.org/records/6647024). You can read more about the dataset in the [rMATS-turbo manuscript](https://pubmed.ncbi.nlm.nih.gov/38396040/).

For simplicity, we include the shell commands for downloading and extracting these necessary files below. You can also alternatively also download and extract these files yourself.

In [ ]:
!wget -O PC3E-GS689.tar https://zenodo.org/records/6647024/files/PC3E-GS689.tar.gz?download=1
!tar -xvzf PC3E-GS689.tar

--2026-01-06 05:32:19--  https://zenodo.org/records/6647024/files/PC3E-GS689.tar.gz?download=1
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 188.185.43.153, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 119609490 (114M) [application/octet-stream]
Saving to: ‘PC3E-GS689.tar’

PC3E-GS689.tar      100%[===================>] 114.07M  1.84MB/s    in 89s     

2026-01-06 05:33:48 (1.29 MB/s) - ‘PC3E-GS689.tar’ saved [119609490/119609490]

PC3E-GS689/
PC3E-GS689/fromGTF.novelJunction.SE.txt
PC3E-GS689/fromGTF.novelSpliceSite.A5SS.txt
PC3E-GS689/SE.MATS.JC.txt
PC3E-GS689/RI.MATS.JCEC.txt
PC3E-GS689/JCEC.raw.input.RI.txt
PC3E-GS689/fromGTF.novelSpliceSite.MXE.txt
PC3E-GS689/fromGTF.RI.txt
PC3E-GS689/JC.raw.input.RI.txt
PC3E-GS689/summary.txt
PC3E-GS689/fromGTF.novelJunction.RI.txt
PC3E-GS689/A3SS.MATS.JCEC.txt
PC3E-GS689/fromGTF.novelSpliceSite.SE.txt
PC3E-GS689/JC.raw.input.SE.txt
PC3E-GS6

In our benchmarking, we run Splitpea on the rMATS SE.MATS.JCEC.txt file and measure wall-clock runtime while running Splitpea on randomly sampled subsets of splicing events (1%, 25%, 50%, 75%, and 100% of the data).

In [ ]:
!lscpu | egrep 'Model name|CPU\(s\)|Thread|Core|Socket|MHz|Vendor'

CPU(s):                                  2
On-line CPU(s) list:                     0,1
Vendor ID:                               AuthenticAMD
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Core(s) per socket:                      1
Socket(s):                               1
NUMA node0 CPU(s):                       0,1


In [ ]:
IN_FILE = "PC3E-GS689/SE.MATS.JCEC.txt"
SEED = 42
FRACTIONS = [0.01, 0.25, 0.50, 0.75, 1.00]

OUT_DIR = "splitpea_sampling_runs"
OUT_PREFIX_BASE = "PC3E_GS689"  # each run will append _pct25/_pct50/_pct75/_pct100
KEEP_HEADER = "auto"

in_path = Path(IN_FILE)
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

random.seed(SEED)

def parse_header(line: str) -> bool:
    if "\t" not in line:
        return False
    fields = line.rstrip("\n").split("\t")
    alpha_fields = sum(any(c.isalpha() for c in f) for f in fields)
    return alpha_fields >= max(2, len(fields)//4)

def read_lines(path: Path, keep_header):
    with path.open("r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()
    header = None
    data = lines
    if keep_header == True and lines:
        header, data = lines[0], lines[1:]
    elif keep_header == "auto" and lines and parse_header(lines[0]):
        header, data = lines[0], lines[1:]
    return header, data

header, data_lines = read_lines(in_path, KEEP_HEADER)
n_total_data = len(data_lines)

rows = []

for frac in FRACTIONS:
    print(frac)
    pct = int(round(frac * 100))
    n_sample = n_total_data if frac >= 1.0 else max(1, int(round(n_total_data * frac)))

    sampled = data_lines if frac >= 1.0 else random.sample(data_lines, n_sample)

    sampled_infile = out_dir / f"{in_path.stem}.sample{pct}.txt"
    with sampled_infile.open("w", encoding="utf-8") as f:
        if header is not None:
            f.write(header if header.endswith("\n") else header + "\n")
        f.writelines(sampled)

    run_prefix = str(out_dir / f"{OUT_PREFIX_BASE}_pct{pct}")

    t0 = time.perf_counter()
    splitpea.run(
        in_file=str(sampled_infile),
        differential_format="rmats",
        out_file_prefix=run_prefix,
        verbose=True
    )
    elapsed_s = time.perf_counter() - t0
    print(elapsed_s)

    rows.append({
        "fraction": frac,
        "percent": pct,
        "n_total_data_lines": n_total_data,
        "n_sampled_data_lines": n_sample,
        "input_file": str(sampled_infile),
        "out_file_prefix": run_prefix,
        "elapsed_seconds": elapsed_s,
        "seed": SEED,
        "kept_header": header is not None
    })

df = pd.DataFrame(rows).sort_values("fraction").reset_index(drop=True)
display(df)

df.to_csv(out_dir / "splitpea_timings.csv", index=False)
df.to_pickle(out_dir / "splitpea_timings.pkl")


0.01


INFO:diff_exon:Loading DDIs....
INFO:diff_exon:# pfams: 8602, # ddis: 17450
INFO:diff_exon:Loading PPIs....
INFO:diff_exon:# proteins: 20286, # interactions: 793078
INFO:diff_exon:Loading gene-protein domain info....
INFO:diff_exon:Reading differential exon results....
INFO:diff_exon:Reading in input exons...
INFO:diff_exon:# of differentially expressed exons: 213
INFO:diff_exon:Calculating network....
INFO:diff_exon:# nodes: 345, # edges: 335
INFO:diff_exon:Outputting as pickle...
INFO:diff_exon:Temporary file /content/tmpqxdqblmq.txt deleted.
INFO:diff_exon:Done


13.242577595
0.25


INFO:diff_exon:Loading DDIs....
INFO:diff_exon:# pfams: 8602, # ddis: 17450
INFO:diff_exon:Loading PPIs....
INFO:diff_exon:# proteins: 20286, # interactions: 793078
INFO:diff_exon:Loading gene-protein domain info....
INFO:diff_exon:Reading differential exon results....
INFO:diff_exon:Reading in input exons...
INFO:diff_exon:# of differentially expressed exons: 5420
INFO:diff_exon:Calculating network....
INFO:diff_exon:# nodes: 3941, # edges: 7606
INFO:diff_exon:Outputting as pickle...
INFO:diff_exon:Temporary file /content/tmpe7lz77nm.txt deleted.
INFO:diff_exon:Done


78.17068046200004
0.5


...

INFO:diff_exon:Loading DDIs....
INFO:diff_exon:# pfams: 8602, # ddis: 17450
INFO:diff_exon:Loading PPIs....
INFO:diff_exon:# proteins: 20286, # interactions: 793078
INFO:diff_exon:Loading gene-protein domain info....
INFO:diff_exon:Reading differential exon results....
INFO:diff_exon:Reading in input exons...
INFO:diff_exon:# of differentially expressed exons: 15538
INFO:diff_exon:Calculating network....
INFO:diff_exon:# nodes: 5818, # edges: 15061
INFO:diff_exon:Outputting as pickle...
INFO:diff_exon:Temporary file /content/tmpq8z4tr55.txt deleted.
INFO:diff_exon:Done


214.79989701500006
1.0


INFO:diff_exon:Loading DDIs....
INFO:diff_exon:# pfams: 8602, # ddis: 17450
INFO:diff_exon:Loading PPIs....
INFO:diff_exon:# proteins: 20286, # interactions: 793078
INFO:diff_exon:Loading gene-protein domain info....
INFO:diff_exon:Reading differential exon results....
INFO:diff_exon:Reading in input exons...
INFO:diff_exon:# of differentially expressed exons: 20067
INFO:diff_exon:Calculating network....
INFO:diff_exon:# nodes: 6173, # edges: 17012
INFO:diff_exon:Outputting as pickle...
INFO:diff_exon:Temporary file /content/tmpbld3t_71.txt deleted.
INFO:diff_exon:Done


284.73288023


,fraction,percent,n_total_data_lines,n_sampled_data_lines,input_file,out_file_prefix,elapsed_seconds,seed,kept_header
0,0.01,1,308739,3087,splitpea_sampling_runs/SE.MATS.JCEC.sample1.txt,splitpea_sampling_runs/PC3E_GS689_pct1,13.242578,42,True
1,0.25,25,308739,77185,splitpea_sampling_runs/SE.MATS.JCEC.sample25.txt,splitpea_sampling_runs/PC3E_GS689_pct25,78.170680,42,True
2,0.50,50,308739,154370,splitpea_sampling_runs/SE.MATS.JCEC.sample50.txt,splitpea_sampling_runs/PC3E_GS689_pct50,146.580380,42,True
3,0.75,75,308739,231554,splitpea_sampling_runs/SE.MATS.JCEC.sample75.txt,splitpea_sampling_runs/PC3E_GS689_pct75,214.799897,42,True
4,1.00,100,308739,308739,splitpea_sampling_runs/SE.MATS.JCEC.sample100.txt,splitpea_sampling_runs/PC3E_GS689_pct100,284.732880,42,True
